In [0]:
from pyspark.sql.functions import col, to_date, expr, current_date

orders_df = spark.table("main.ecommerce.orders")
order_items_df = spark.table("main.ecommerce.order_items")
customers_df = spark.table("main.ecommerce.customers")
products_df = spark.table("main.ecommerce.products")
order_payments_df = spark.table("main.ecommerce.order_payments")

# --> Join operation
fact_df = order_items_df \
    .join(orders_df, "order_id") \
    .join(customers_df, "customer_id") \
    .join(products_df, "product_id") \
    .join(order_payments_df, "order_id")

# --> Required columns
#Identifiers
fact_sales_df = fact_df.select(col("order_id"), col("order_item_id"),col("customer_id"),col("product_id"),
#Time Attributes
col("order_purchase_timestamp"), to_date(col("order_purchase_timestamp")).alias("order_date"),
#Measures
col("price"),col("freight_value"), expr("price + freight_value").alias("revenue"),
#Dimensional Attributes
col("customer_state"),col("product_category_name"),
#Optional
col("payment_value"), col("order_status"), current_date().alias("load_date")
)


# --> create fact table (delta)
fact_sales_df.write.format("delta").mode("overwrite").saveAsTable("main.ecommerce.fact_sales")


In [0]:
display(fact_sales_df)